In [1]:
import pymorphy3
import string
import ahocorasick

import pandas as pd
from corus import load_lenta
from collections import defaultdict
from itertools import product

In [5]:
morph = pymorphy3.MorphAnalyzer()

In [6]:
def get_all_forms(word):
    forms = set()
    for parse in morph.parse(word):
        for inflected in parse.lexeme:
            forms.add(inflected.word)
    return forms

In [7]:
def get_idiom_variants(idiom):
    words = idiom.lower().split()
    all_word_forms = [get_all_forms(w) for w in words]
    variants = set()
    for combo in product(*all_word_forms):
        variants.add(" ".join(combo))
    return variants

In [8]:
path_lenta = 'lenta-ru-news.csv.gz'

In [9]:
with open('idioms_list.txt', 'r', encoding='utf-8') as f:
    idioms_list = [line.strip().lower() for line in f if line.strip()]

In [11]:
table = str.maketrans('', '', string.punctuation)
idioms_list = [s.translate(table) for s in idioms_list]

In [ ]:
variant_to_idiom = {}
for idiom in idioms_list:
    for variant in get_idiom_variants(idiom):
        variant_to_idiom[variant] = idiom

automaton = ahocorasick.Automaton()
for variant, idiom in variant_to_idiom.items():
    automaton.add_word(variant, (variant, idiom))
automaton.make_automaton()

In [14]:
MAX_PER_IDIOM = 5
counts = defaultdict(int)
dataset = []

for record in load_lenta(path_lenta):
    text_lower = record.text.lower()

    if all(counts[i] >= MAX_PER_IDIOM for i in idioms_list):
        break

    found_idioms = set()
    for _, (variant, idiom) in automaton.iter(text_lower):
        if counts[idiom] < MAX_PER_IDIOM:
            found_idioms.add(idiom)

    if not found_idioms:
        continue

    for sent in record.text.split('.'):
        sent_lower = sent.lower()
        for _, (variant, idiom) in automaton.iter(sent_lower):
            if idiom in found_idioms and counts[idiom] < MAX_PER_IDIOM:
                dataset.append({'idiom': idiom, 'context': sent.strip()})
                counts[idiom] += 1
                found_idioms.discard(idiom)

df = pd.DataFrame(dataset)
df = df[df['idiom'].map(counts) >= MAX_PER_IDIOM]
df.to_csv('idioms_examples.csv', index=False, encoding='utf-8')